# Step 3: Advanced Ranking & Behavioral Heuristics

In Phase 2, FAISS gave us candidates that were semantically closest to the Job Description. But as you saw, they might just be random roles, or worse, they could be **Keyword Stuffers** or **Ghosters**!

In Phase 3, we have introduced a **Behavioral Multiplier** that penalizes candidates who ghost recruiters, skip interviews, or repetitively stuff keywords like "AI" into their profile to trick the algorithm.

Let's see how our final ranking algorithm shifts the FAISS baseline!

In [1]:
import sys
import pandas as pd
sys.path.append('../src')

from data_ingestion import load_candidates
from features import stringify_candidate
from embeddings import EmbeddingModel, CandidateFAISS
from ranking import calculate_final_scores

# 1. Load Data
sample_path = '../[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/sample_candidates.json'
candidates = load_candidates(sample_path)
candidate_strings = [stringify_candidate(c) for c in candidates]

Successfully loaded 50 candidates from sample_candidates.json


### Generate Baseline Embeddings

In [2]:
# 2. Embed and Index
embedder = EmbeddingModel('all-MiniLM-L6-v2')
faiss_index = CandidateFAISS(embedding_dim=384)
candidate_embeddings = embedder.generate_embeddings(candidate_strings)
faiss_index.add_embeddings(candidate_embeddings)

Loading SentenceTransformer model: all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings for 50 texts...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Added 50 vectors to FAISS index. Total: 50


### Apply the Behavioral Heuristics Multiplier

We pull the top 15 FAISS candidates and then re-rank them using our heuristic multiplier from `src/heuristics.py`.

In [4]:
# 3. Load and Embed Job Description
jd_path = '../[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/job_description.docx.txt'
with open(jd_path, 'r', encoding='utf-8') as f:
    job_description = f.read()
    
jd_embedding = embedder.generate_embeddings([job_description])[0]

# 4. Get Top 15 from FAISS
top_k = 15
distances, indices = faiss_index.search(jd_embedding, k=top_k)

# 5. Calculate Final Scores (Semantic * Behavioral)
final_rankings = calculate_final_scores(candidates, distances, indices)

# Display Top 10
print("\n--- FINAL ADVANCED RANKINGS (TOP 10) ---")
for i, res in enumerate(final_rankings[:10]):
    candidate = res['candidate']
    name = candidate.get('profile', {}).get('anonymized_name', 'Unknown')
    title = candidate.get('profile', {}).get('current_title', 'No Title')
    score = res['final_score']
    sem = res['semantic_score']
    mult = res['multiplier']
    
    print(f"{i+1}. {name} | {title}")
    print(f"   -> Final Score: {score:.4f} (Semantic: {sem:.4f}, Multiplier: {mult:.2f})\n")

Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


--- FINAL ADVANCED RANKINGS (TOP 10) ---
1. Tanya Chowdary | Mechanical Engineer
   -> Final Score: 0.3514 (Semantic: 0.5430, Multiplier: 0.65)

2. Sai Saxena | Marketing Manager
   -> Final Score: 0.3410 (Semantic: 0.5513, Multiplier: 0.62)

3. Amit Shah | Mechanical Engineer
   -> Final Score: 0.3396 (Semantic: 0.5419, Multiplier: 0.63)

4. Saanvi Sethi | Operations Manager
   -> Final Score: 0.2994 (Semantic: 0.5462, Multiplier: 0.55)

5. Shaurya Chatterjee | Operations Manager
   -> Final Score: 0.2902 (Semantic: 0.5381, Multiplier: 0.54)

6. Naina Bose | Business Analyst
   -> Final Score: 0.2849 (Semantic: 0.5398, Multiplier: 0.53)

7. Anjali Khanna | Operations Manager
   -> Final Score: 0.2678 (Semantic: 0.5489, Multiplier: 0.49)

8. Rahul Joshi | Project Manager
   -> Final Score: 0.1812 (Semantic: 0.5625, Multiplier: 0.32)

9. Rajesh Desai | Business Analyst
   -> Final Score: 0.1810 (Semantic: 0.5561, Multiplier: 0.33)

10. Neha Rao | Graphic Designer
   -> Final Score: 0.1